# Introduction


This Notebook introduces Gemma 4:E2B (it).

The new model from Gemma series arrives in 4 parameter sizes:
* E2B
* E4B
* 26B
* 31B

We will test the multimodal and multilanguage capability of the most compact model E2B(it).

All models accept multimodal input (text, image, video) and output text.



# Upgrade transformers

In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 88.8 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


# Define a pipeline

In [2]:
from transformers import pipeline
pipe = pipeline("any-to-any", model="google/gemma-4-e2b-it")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

# Test with image

Let's use here the example from HuggingFace.
We will analyze an image.

[](https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/thailand.jpg)

<img src="https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/thailand.jpg"></img>

In [3]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/thailand.jpg",
            },
            {"type": "text", "text": "Do you have travel advice going to here?"},
        ],
    }
]
output = pipe(messages, max_new_tokens=100, return_full_text=False)
output[0]["generated_text"]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"Based on the image, you are looking at a magnificent, ornate **Buddhist temple or pagoda**, likely in **Southeast Asia**, given the architectural style (which strongly resembles temples in countries like Thailand, Myanmar, or Laos).\n\nSince the image itself doesn't provide a specific location, I can only offer **general travel advice** based on what this type of destination typically entails.\n\n**To give you specific and helpful advice, please tell me:**\n\n1. **Where is this place?**"

Let's beautify a bit the output.

In [4]:
from IPython.display import Markdown

display(Markdown(output[0]["generated_text"]))

Based on the image, you are looking at a magnificent, ornate **Buddhist temple or pagoda**, likely in **Southeast Asia**, given the architectural style (which strongly resembles temples in countries like Thailand, Myanmar, or Laos).

Since the image itself doesn't provide a specific location, I can only offer **general travel advice** based on what this type of destination typically entails.

**To give you specific and helpful advice, please tell me:**

1. **Where is this place?**

As we limited the number of tokens to 100, the output message is not fully displayed.

# Test with video


We continue now also with the video from HuggingFace example.

Let's first display the video.

In [5]:
from IPython.display import Video

Video("https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/rockets.mp4")

In [6]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": "https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/rockets.mp4",
            },
            {"type": "text", "text": "What is happening in this video?"},
        ],
    }
]

output = pipe(messages, load_audio_from_video=True)
output[0]["generated_text"]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'role': 'user',
  'content': [{'type': 'video',
    'video': 'https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/rockets.mp4'},
   {'type': 'text', 'text': 'What is happening in this video?'},
   {'type': 'audio'}]},
 {'role': 'assistant',
  'content': 'This video shows a large crowd of people gathered on a tarmac, watching a massive rocket, which appears to be a SpaceX Falcon 9 rocket, being prepared or launched. The sky is cloudy, suggesting either dawn, dusk, or an overcast day. There are also other aircraft and structures visible in the background, including the France airline branding. The atmosphere seems excited, as people are looking up at the rocket.'}]

Let's show the answer a bit beautified.

In [7]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

This video shows a large crowd of people gathered on a tarmac, watching a massive rocket, which appears to be a SpaceX Falcon 9 rocket, being prepared or launched. The sky is cloudy, suggesting either dawn, dusk, or an overcast day. There are also other aircraft and structures visible in the background, including the France airline branding. The atmosphere seems excited, as people are looking up at the rocket.

Not sure how accurate is the answer, since the rocket shows Arianne logo, but the answer is close enough.

# More tests


Let's try also with another images.


## A cow on the beach

<img src="https://storage.googleapis.com/keras-cv/models/paligemma/cow_beach_1.png"></img>

In [8]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://storage.googleapis.com/keras-cv/models/paligemma/cow_beach_1.png"},
            {"type": "text", "text": "What you can see in this image?"}
        ]
    }
]

output = pipe(text=messages, max_new_tokens=300)
print(output[0]["generated_text"][-1]["content"])

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In this image, I can see a **brown and white cow** standing on a **sandy beach**.

Here are the details:

*   **Subject:** The main subject is a bovine animal, likely a cow, with rich brown fur and a prominent white patch on its face.
*   **Setting (Foreground/Midground):** The cow is standing on light-colored sand. There appears to be some wet sand or a slight puddle near its front legs, suggesting the tide might be low or the area is damp.
*   **Setting (Background):** In the background, there is a **calm ocean or sea** with turquoise or light blue water. Further in the distance, there are **landmasses or hills** on the horizon.
*   **Atmosphere:** The scene appears bright, suggesting a **sunny day** with a clear **blue sky** and some scattered white clouds.

Overall, it is a picturesque outdoor scene featuring livestock by the sea.


In [9]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

In this image, I can see a **brown and white cow** standing on a **sandy beach**.

Here are the details:

*   **Subject:** The main subject is a bovine animal, likely a cow, with rich brown fur and a prominent white patch on its face.
*   **Setting (Foreground/Midground):** The cow is standing on light-colored sand. There appears to be some wet sand or a slight puddle near its front legs, suggesting the tide might be low or the area is damp.
*   **Setting (Background):** In the background, there is a **calm ocean or sea** with turquoise or light blue water. Further in the distance, there are **landmasses or hills** on the horizon.
*   **Atmosphere:** The scene appears bright, suggesting a **sunny day** with a clear **blue sky** and some scattered white clouds.

Overall, it is a picturesque outdoor scene featuring livestock by the sea.

# Small detail on a candy


<img src="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"></img>

In [10]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is represented on the candy?"}
        ]
    }
]
output = pipe(text=messages, max_new_tokens=200)


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [11]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

Based on the image, the candies appear to be **ladybugs**.

They are small, round, and feature colors (green, orange) and markings that resemble the pattern of a ladybug.

Actually, the small drawing on the candies are more like turtles.

Let's check now both the counting abilities of this compact model as well as German language knowledge.

In [12]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "Welche Farben haben die Bombons?"}
        ]
    }
]
output = pipe(text=messages, max_new_tokens=200)


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [13]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

Die Bombons auf dem Bild haben folgende Farben:

* **Türkisgrün/Blaugrün** (zwei Stück)
* **Orange** (eins)
* **Grün** (eins)

The answer is perfect.

# Spanish culture and language


<img src="https://d1bv4heaa2n05k.cloudfront.net/user-images/1439905381602/shutterstock-78898486_destinationMain_1439905420657.jpeg"></img>

In [14]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://d1bv4heaa2n05k.cloudfront.net/user-images/1439905381602/shutterstock-78898486_destinationMain_1439905420657.jpeg"},
            {"type": "text", "text": "¿Qué ves en esta imagen?"}
        ]
    }
]
output = pipe(text=messages, max_new_tokens=200)


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [15]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

En la imagen se observa una escena que parece ser de una **exhibición, evento cultural o una competencia**, probablemente relacionada con la cultura taurina o alguna tradición similar.

Aquí tienes un desglose de lo que se ve:

1.  **Una mujer vestida con un traje llamativo:**
    *   Lleva un traje muy elaborado, con detalles dorados y patrones intrincados (posiblemente bordados o con lentejuelas) en la parte superior.
    *   Viste pantalones ajustados de color oscuro (negro o morado oscuro).
    *   Tiene mangas largas y detalles brillantes.
    *   Lleva un **velo o capa grande de color púrpura o magenta vibrante** que se extiende dramáticamente a su alrededor.
    *   Sus piernas y pies están cubiertos por medias o botas de color rosa fuerte.
    *   Su pose es dinámica, sugiriendo movimiento o actuación.

2.

The model is smart, but not that smart. It can interpret all the features and even small details in the image, but it is not recognizing a torrero scene.

# Japanese landmark

<img src="https://www.advantour.com/img/japan/tokyo/tokyo-tower.jpg"></img>

In [16]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://www.advantour.com/img/japan/tokyo/tokyo-tower.jpg"},
            {"type": "text", "text": "この画像には何が見えますか?"}
        ]
    }
]
output = pipe(text=messages, max_new_tokens=200)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [17]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

この画像は、**晴れた日の都市のスカイライン**を捉えたものです。

具体的に見られる要素は以下の通りです。

1. **スカイラインと高層ビル群:**
    * 遠景には、多くの高層ビルが密集した都市の風景が広がっています。
    * 中央から右側にかけて、現代的なガラス張りの高層ビル群が見えます。
    * 特に目立つのは、赤や緑などの色をした特徴的な高層ビルです。

2. **東京スカイツリー:**
    * 画像の左中央に、非常に高い**東京スカイツリー**がそびえ立っています。その赤い構造が空に対して際立っています。

3. **自然の要素（緑）:**
    * 都市の風景の中にも、**豊かな緑（木々）**が多く見られます。特に手前や中景には、紅葉しているか、緑の葉を持つ公園

The model was able to interpret correctly a landmark image from Tokyo, Tokyo Tower, with all details.

# French landmark

<img src="https://wmf.imgix.net/images/aa_fra_notre-dame_de_paris_0.jpg"></img>


In [18]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://wmf.imgix.net/images/aa_fra_notre-dame_de_paris_0.jpg"},
            {"type": "text", "text": "Que voyez-vous sur cette photo? Repondez succinte, svp."}
        ]
    }
]
output = pipe(text=messages, max_new_tokens=200)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [19]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

Cette photo montre la **Cathédrale Notre-Dame de Paris** (ou une cathédrale similaire de style gothique) sous un ciel bleu. On aperçoit également une place animée avec des bâtiments anciens et des arbres sur la droite.

The answer is quite good, recognizing the Notre Dame cathedral.

# Conclusions

We checked the multimodal and multilanguage features of Gemma 3:2B (it) model.

The model can interpret both image and video.

We could verify that the model is able to interpret correctly a variety of images (describe the content of an inedite scene, perceive small details in a picture, correctly identify a landmark) and is also capable to process the text (and output answer) in multiple languages. 

Being a small size model (2B), it miss some of the contextual information (e.g. did not recognized a corrida) but in general it is impressive by the capacity to deal with multi-modal and multi-language data.

We used English, German, French, Spanish, and Japanese.